In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import time
import psutil
from pyspark.sql import SparkSession
import numpy as np
from pyspark.sql import Row
from pyspark.sql import SparkSession
from redis.cluster import RedisCluster, ClusterNode

In [2]:
memory_use = "16g"
spark = (SparkSession.builder
    .appName("Optimized_RF_50M_Rows")
    # Allocate 12GB for drivers, leaving 4GB for the OS and Docker overhead.
    .config("spark.driver.memory", memory_use)
    .config("spark.executor.memory", memory_use)
    
    # Memory Optimization: Allocate 80% of RAM to Spark,
    # Prioritizing Execution (RF computation) over Storage (cache)
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Data partitioning: 50 million rows should be divided into at least 500 partitions.
    .config("spark.sql.shuffle.partitions", "500")
    .config("spark.default.parallelism", "500")
    
    # Avoid timeouts when processing large volumes.
    .config("spark.network.timeout", "1200s")
    .config("spark.sql.broadcastTimeout", "1200s")

    .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 19:46:10 WARN Utils: Your hostname, lenovo-Legion-5-15IAH7H, resolves to a loopback address: 127.0.1.1; using 192.168.1.8 instead (on interface wlp0s20f3)
26/05/06 19:46:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 19:46:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 19:46:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
# def count_redis_records_spark(pattern="drug:*"):
#     spark = SparkSession.builder.appName("RedisCount").getOrCreate()
#     sc = spark.sparkContext

#     # 1. Khởi tạo kết nối tạm thời trên Driver để lấy danh sách Master Nodes
#     startup_nodes = [ClusterNode("172.25.0.10", 6379)]
#     rc_driver = RedisCluster(startup_nodes=startup_nodes, password='1234', decode_responses=True)
    
#     # Lấy danh sách các node Master (8 shards)
#     # Kết quả trả về dạng danh sách các dict chứa thông tin host, port
#     primaries = [{"host": n.host, "port": n.port} for n in rc_driver.get_primaries()]
    
#     # 2. Tạo một RDD từ danh sách các Master nodes
#     # Mỗi node sẽ là một phần tử trong RDD
#     nodes_rdd = sc.parallelize(primaries, len(primaries))

#     # 3. Hàm thực thi trên mỗi Executor
#     def count_in_node(node_info):
#         # Mỗi executor thiết lập kết nối riêng tới shard đó
#         from redis.cluster import RedisCluster, ClusterNode
        
#         node = ClusterNode(node_info['host'], node_info['port'])
#         # Kết nối tới cluster nhưng chỉ định mục tiêu là node hiện tại
#         rc_executor = RedisCluster(startup_nodes=[node], password='1234', decode_responses=True)
        
#         # Sử dụng scan_iter để đếm key khớp pattern
#         scanner = rc_executor.scan_iter(match=pattern, target_nodes=[node])
#         count = sum(1 for _ in scanner)
        
#         return count

#     # 4. Map các node tới số lượng đếm được và tính tổng (Reduce)
#     total_count = nodes_rdd.map(count_in_node).sum()
    
#     return total_count

def count_redis_records_distributed(pattern="drug:*"):
    # 1. Khởi tạo Spark Session
    spark = SparkSession.builder.appName("RedisFastCount").getOrCreate()
    sc = spark.sparkContext

    # 2. Lấy danh sách các node Master từ Driver
    # Thay đổi IP/Pass cho đúng cấu hình của bạn
    startup_nodes = [ClusterNode("172.25.0.10", 6379)]
    rc_driver = RedisCluster(startup_nodes=startup_nodes, password='1234', decode_responses=False)
    
    # Lấy thông tin các shard Master
    primaries = [{"host": n.host, "port": n.port} for n in rc_driver.get_primaries()]
    
    # 3. Tạo RDD từ danh sách các shard (8 shards = 8 partitions)
    nodes_rdd = sc.parallelize(primaries, len(primaries))

    # 4. Định nghĩa hàm thực thi trên mỗi Executor (Dùng LUA để tối ưu)
    def count_in_node(node_info):
        from redis.cluster import RedisCluster, ClusterNode
        
        # Kết nối đến shard cụ thể
        node = ClusterNode(node_info['host'], node_info['port'])
        # decode_responses=False để tăng tốc độ (không tốn CPU convert string)
        rc_executor = RedisCluster(startup_nodes=[node], password='1234', decode_responses=False)
        
        # Lua script: Đếm trực tiếp trên RAM của Redis shard
        lua_script = """
        local cursor = "0"
        local count = 0
        repeat
            local res = redis.call("SCAN", cursor, "MATCH", ARGV[1], "COUNT", 10000)
            cursor = res[1]
            count = count + #res[2]
        until cursor == "0"
        return count
        """
        
        # Chạy script trên node hiện tại
        # Lưu ý: target_nodes=[node] đảm bảo script chỉ chạy trên đúng shard đó
        node_obj = rc_executor.get_node(node_id=None, host=node_info['host'], port=node_info['port'])
        count = rc_executor.get_connection_by_node(node_obj).eval(lua_script, 0, pattern)
        
        return count

    # 5. Kích hoạt tính toán song song trên Spark
    # Mỗi node trong 8 shards sẽ được một Executor đảm nhiệm đếm cùng lúc
    total_count = nodes_rdd.map(count_in_node).sum()
    
    return total_count

In [ ]:
# Gọi hàm
total = count_redis_records_distributed("drug:*")
print(f"TỔNG SỐ BẢN GHI TRÊN 8 SHARDS: {total}")

26/05/06 19:46:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


TỔNG SỐ BẢN GHI TRÊN 8 SHARDS: 50000000
